In [ ]:
# =========================================================
# FULL MURA PYTORCH PIPELINE — SINGLE CELL
# DenseNet-121 | train_flat / valid_flat
# =========================================================

import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import roc_auc_score, accuracy_score
from PIL import Image
import matplotlib.pyplot as plt

# -------------------------------
# CONFIG
# -------------------------------
train_dir = r"E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\train_flat"
val_dir   = r"E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\valid_flat"

batch_size = 16
epochs_stage1 = 5
epochs_stage2 = 3
lr1 = 1e-4
lr2 = 1e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------
# REMOVE INVALID IMAGES
# -------------------------------
def remove_invalid_images(directory):
    removed = 0
    for root, _, files in os.walk(directory):
        for f in files:
            path = os.path.join(root, f)
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception:
                try:
                    os.remove(path)
                    removed += 1
                except:
                    pass
    print(f"Removed {removed} invalid images from {directory}")

remove_invalid_images(train_dir)
remove_invalid_images(val_dir)

# -------------------------------
# TRANSFORMS
# -------------------------------
train_tf = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# -------------------------------
# DATASETS + LOADERS
# -------------------------------
train_ds = ImageFolder(train_dir, transform=train_tf)
val_ds   = ImageFolder(val_dir,   transform=val_tf)

print("Class mapping:", train_ds.class_to_idx)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

# -------------------------------
# MODEL
# -------------------------------
model = models.densenet121(
    weights=models.DenseNet121_Weights.DEFAULT
)

model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.classifier.in_features, 1)
)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr1)

# -------------------------------
# TRAIN / EVAL FUNCTIONS
# -------------------------------
def train_epoch():
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(train_loader.dataset)

def evaluate():
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            preds.extend(outputs)
            gts.extend(labels.numpy())
    preds = np.array(preds).ravel()
    gts = np.array(gts)
    return roc_auc_score(gts, preds), accuracy_score(gts, preds > 0.5)

# -------------------------------
# STAGE 1 — FROZEN BACKBONE
# -------------------------------
for p in model.features.parameters():
    p.requires_grad = False

for e in range(epochs_stage1):
    loss = train_epoch()
    auc, acc = evaluate()
    print(f"[Stage 1][{e+1}] Loss={loss:.4f} | AUC={auc:.3f} | Acc={acc:.3f}")

# -------------------------------
# STAGE 2 — FINE-TUNING
# -------------------------------
for p in model.features.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=lr2)

for e in range(epochs_stage2):
    loss = train_epoch()
    auc, acc = evaluate()
    print(f"[Stage 2][{e+1}] Loss={loss:.4f} | AUC={auc:.3f} | Acc={acc:.3f}")

# -------------------------------
# SAVE MODEL
# -------------------------------
torch.save(model.state_dict(), "densenet121_mura.pth")
print("✅ Model saved as densenet121_mura.pth")


Using device: cpu
Removed 0 invalid images from E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\train_flat
Removed 0 invalid images from E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\valid_flat
Class mapping: {'abnormal': 0, 'normal': 1}
Train samples: 36808
Val samples: 3197
[Stage 1][1] Loss=0.6670 | AUC=0.669 | Acc=0.553
[Stage 1][2] Loss=0.6426 | AUC=0.697 | Acc=0.571
[Stage 1][3] Loss=0.6335 | AUC=0.711 | Acc=0.575
[Stage 1][4] Loss=0.6298 | AUC=0.716 | Acc=0.576
[Stage 1][5] Loss=0.6268 | AUC=0.719 | Acc=0.621
[Stage 2][1] Loss=0.5263 | AUC=0.861 | Acc=0.769
[Stage 2][2] Loss=0.4703 | AUC=0.866 | Acc=0.802
[Stage 2][3] Loss=0.4462 | AUC=0.876 | Acc=0.804
✅ Model saved as densenet121_mura.pth


In [ ]:
# =========================================================
# FINAL MURA PYTORCH PIPELINE — CPU OPTIMIZED (SINGLE CELL)
# DenseNet-121 | Class-weighted loss | Proper fine-tuning
# =========================================================

import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import roc_auc_score, accuracy_score
from PIL import Image

# -------------------------------
# CONFIG
# -------------------------------
train_dir = r"E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\train_flat"
val_dir   = r"E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\valid_flat"

batch_size = 16
epochs_stage1 = 5
epochs_stage2 = 6
lr1 = 1e-4
lr2 = 3e-5

device = torch.device("cpu")
print("Using device:", device)

# -------------------------------
# REMOVE INVALID IMAGES
# -------------------------------
def remove_invalid_images(directory):
    removed = 0
    for root, _, files in os.walk(directory):
        for f in files:
            path = os.path.join(root, f)
            try:
                with Image.open(path) as img:
                    img.verify()
            except:
                try:
                    os.remove(path)
                    removed += 1
                except:
                    pass
    print(f"Removed {removed} invalid images from {directory}")

remove_invalid_images(train_dir)
remove_invalid_images(val_dir)

# -------------------------------
# TRANSFORMS (CPU FRIENDLY)
# -------------------------------
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.25),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# -------------------------------
# DATASETS + LOADERS
# -------------------------------
train_ds = ImageFolder(train_dir, transform=train_tf)
val_ds   = ImageFolder(val_dir,   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

print("Class mapping:", train_ds.class_to_idx)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

# -------------------------------
# CLASS WEIGHT (CRITICAL)
# -------------------------------
labels = [y for _, y in train_ds]
pos = sum(labels)
neg = len(labels) - pos
pos_weight = torch.tensor([neg / pos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -------------------------------
# MODEL
# -------------------------------
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.classifier.in_features, 1)
)

model = model.to(device)

# -------------------------------
# TRAIN / EVAL FUNCTIONS
# -------------------------------
def train_epoch():
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

    return total_loss / len(train_loader.dataset)

def evaluate():
    model.eval()
    preds, gts = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            preds.extend(outputs)
            gts.extend(labels.numpy())

    preds = np.array(preds).ravel()
    gts = np.array(gts)

    auc = roc_auc_score(gts, preds)

    thresholds = np.linspace(0.1, 0.9, 81)
    best_acc, best_t = 0, 0.5
    for t in thresholds:
        acc = accuracy_score(gts, preds > t)
        if acc > best_acc:
            best_acc, best_t = acc, t

    return auc, best_acc, best_t

# -------------------------------
# STAGE 1 — FROZEN BACKBONE
# -------------------------------
for p in model.features.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=lr1,
    weight_decay=1e-4
)

print("\n--- STAGE 1: Training classifier only ---")
for e in range(epochs_stage1):
    loss = train_epoch()
    auc, acc, t = evaluate()
    print(f"[Stage1][{e+1}] Loss={loss:.4f} | AUC={auc:.3f} | Acc={acc:.3f} | Thr={t:.2f}")

# -------------------------------
# STAGE 2 — PARTIAL + FULL FINE-TUNING
# -------------------------------
for p in model.features.denseblock4.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr2,
    weight_decay=1e-4
)

print("\n--- STAGE 2: Fine-tuning ---")
for e in range(epochs_stage2):
    loss = train_epoch()
    auc, acc, t = evaluate()
    print(f"[Stage2][{e+1}] Loss={loss:.4f} | AUC={auc:.3f} | Acc={acc:.3f} | Thr={t:.2f}")

# -------------------------------
# SAVE MODEL
# -------------------------------
torch.save(model.state_dict(), "densenet121_mura_best_cpu.pth")
print("\n✅ FINAL MODEL SAVED: densenet121_mura_best_cpu.pth")
# =========================================================

In [2]:
# =========================================================
# FINAL MURA PYTORCH PIPELINE — MAX ACCURACY (CPU ONLY)
# DenseNet-121 | Augmentation | MixUp | Label Smoothing
# =========================================================

import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import roc_auc_score, accuracy_score
from PIL import Image

# -------------------------------
# CONFIG
# -------------------------------
train_dir = r"E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\train_flat"
val_dir   = r"E:\grad project 1\medical agent\data sets\xray of bones\MURA-v1.1\MURA-v1.1\valid_flat"

batch_size = 16
epochs_stage1 = 5
epochs_stage2 = 7
lr1 = 1e-4
lr2 = 3e-5
label_smoothing = 0.05
mixup_alpha = 0.2

device = torch.device("cpu")
print("Using device:", device)

# -------------------------------
# REMOVE INVALID IMAGES
# -------------------------------
def remove_invalid_images(directory):
    for root, _, files in os.walk(directory):
        for f in files:
            path = os.path.join(root, f)
            try:
                with Image.open(path) as img:
                    img.verify()
            except:
                try: os.remove(path)
                except: pass

remove_invalid_images(train_dir)
remove_invalid_images(val_dir)

# -------------------------------
# TRANSFORMS (X-RAY OPTIMIZED)
# -------------------------------
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.2, contrast=0.35)
    ], p=0.8),
    transforms.RandomAutocontrast(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# -------------------------------
# DATASETS + LOADERS
# -------------------------------
train_ds = ImageFolder(train_dir, transform=train_tf)
val_ds   = ImageFolder(val_dir,   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

# -------------------------------
# CLASS WEIGHT
# -------------------------------
labels = [y for _, y in train_ds]
pos = sum(labels)
neg = len(labels) - pos
pos_weight = torch.tensor([neg / pos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -------------------------------
# MODEL
# -------------------------------
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.classifier.in_features, 1)
)
model = model.to(device)

# -------------------------------
# UTILITIES
# -------------------------------
def smooth_labels(y):
    return y * (1 - label_smoothing) + 0.5 * label_smoothing

def mixup(x, y):
    lam = np.random.beta(mixup_alpha, mixup_alpha)
    idx = torch.randperm(x.size(0))
    return lam*x + (1-lam)*x[idx], lam*y + (1-lam)*y[idx]

# -------------------------------
# TRAIN / EVAL
# -------------------------------
def train_epoch():
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        imgs, labels = mixup(imgs, labels)
        labels = smooth_labels(labels)

        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

    return total_loss / len(train_loader.dataset)

def evaluate():
    model.eval()
    preds, gts = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            preds.extend(outputs)
            gts.extend(labels.numpy())

    preds = np.array(preds).ravel()
    gts = np.array(gts)

    auc = roc_auc_score(gts, preds)

    thresholds = np.linspace(0.1, 0.9, 81)
    best_acc, best_t = 0, 0.5
    for t in thresholds:
        acc = accuracy_score(gts, preds > t)
        if acc > best_acc:
            best_acc, best_t = acc, t

    return auc, best_acc, best_t

# -------------------------------
# STAGE 1 — CLASSIFIER ONLY
# -------------------------------
for p in model.features.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=lr1,
    weight_decay=1e-4
)

print("\n--- STAGE 1 ---")
for e in range(epochs_stage1):
    loss = train_epoch()
    auc, acc, t = evaluate()
    print(f"[S1][{e+1}] Loss={loss:.4f} | AUC={auc:.3f} | Acc={acc:.3f} | Thr={t:.2f}")

# -------------------------------
# STAGE 2 — FINE-TUNING
# -------------------------------
for p in model.features.denseblock4.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr2,
    weight_decay=1e-4
)

print("\n--- STAGE 2 ---")
for e in range(epochs_stage2):
    loss = train_epoch()
    auc, acc, t = evaluate()
    print(f"[S2][{e+1}] Loss={loss:.4f} | AUC={auc:.3f} | Acc={acc:.3f} | Thr={t:.2f}")

# -------------------------------
# SAVE
# -------------------------------
torch.save(model.state_dict(), "densenet121_mura_max_accuracy_cpu.pth")
print("\n✅ SAVED: densenet121_mura_max_accuracy_cpu.pth")


Using device: cpu

--- STAGE 1 ---
[S1][1] Loss=0.5546 | AUC=0.695 | Acc=0.649 | Thr=0.46
[S1][2] Loss=0.5359 | AUC=0.722 | Acc=0.669 | Thr=0.50
[S1][3] Loss=0.5302 | AUC=0.726 | Acc=0.673 | Thr=0.51
[S1][4] Loss=0.5295 | AUC=0.737 | Acc=0.677 | Thr=0.50
[S1][5] Loss=0.5282 | AUC=0.733 | Acc=0.674 | Thr=0.46

--- STAGE 2 ---
[S2][1] Loss=0.4944 | AUC=0.822 | Acc=0.753 | Thr=0.57
[S2][2] Loss=0.4684 | AUC=0.839 | Acc=0.772 | Thr=0.54
[S2][3] Loss=0.4552 | AUC=0.841 | Acc=0.778 | Thr=0.55
[S2][4] Loss=0.4484 | AUC=0.849 | Acc=0.784 | Thr=0.51
[S2][5] Loss=0.4399 | AUC=0.845 | Acc=0.781 | Thr=0.51
[S2][6] Loss=0.4322 | AUC=0.848 | Acc=0.785 | Thr=0.46
[S2][7] Loss=0.4237 | AUC=0.847 | Acc=0.782 | Thr=0.46

✅ SAVED: densenet121_mura_max_accuracy_cpu.pth
